# Chapter 5.2: SGLang Runtime — Server Architecture & Router

## Learning Objectives

By the end of this notebook, you will:

1. **Understand the SGLang Runtime (SRT)** architecture and its multi-process design
2. **Trace the server launch flow** from `launch_server()` to a fully operational server
3. **Understand the Router** and how requests are distributed to workers
4. **Examine the DataParallelController** for multi-GPU data parallelism
5. **Explore the FastAPI layer** and its OpenAI-compatible endpoints
6. **Follow a complete request** through the runtime pipeline

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part5/chapter_5.2_runtime.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part5/chapter_5.2_runtime.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. SGLang Runtime (SRT) Overview

The SGLang Runtime (SRT) is the inference engine that actually serves LLM requests. It is designed as a **multi-process** system where each component runs in its own process for maximum throughput.

### Architecture Diagram

```
                           ┌─────────────────────────────────┐
                           │         HTTP Clients             │
                           └──────────────┬──────────────────┘
                                          │
                                    HTTP/WebSocket
                                          │
                           ┌──────────────▼──────────────────┐
                           │       FastAPI Server             │
                           │   (uvicorn, main process)        │
                           │                                  │
                           │   /v1/chat/completions           │
                           │   /v1/completions                │
                           │   /generate                     │
                           │   /health                       │
                           └──────────────┬──────────────────┘
                                          │
                                     ZMQ IPC
                                          │
                  ┌───────────────────────▼────────────────────────┐
                  │              TokenizerManager                   │
                  │         (separate process)                     │
                  │  - Tokenize incoming text                      │
                  │  - Apply chat templates                        │
                  │  - Handle image preprocessing (multimodal)     │
                  └───────────────────────┬────────────────────────┘
                                          │
                             ┌────────────▼───────────┐
                             │   DataParallelRouter   │
                             │   (if dp_size > 1)     │
                             └──┬──────────────────┬──┘
                                │                  │
                     ┌──────────▼──────┐  ┌───────▼───────────┐
                     │  Scheduler #0   │  │   Scheduler #1    │
                     │  (dp_rank=0)    │  │   (dp_rank=1)     │
                     │  ┌───────────┐  │  │   ┌───────────┐  │
                     │  │RadixCache │  │  │   │RadixCache │  │
                     │  │MemPool    │  │  │   │MemPool    │  │
                     │  │TpWorker(s)│  │  │   │TpWorker(s)│  │
                     │  └───────────┘  │  │   └───────────┘  │
                     └──────────┬──────┘  └───────┬───────────┘
                                │                  │
                             ┌──▼──────────────────▼──┐
                             │   DetokenizerManager   │
                             │   (separate process)   │
                             └────────────┬───────────┘
                                          │
                                     Response back
                                     to FastAPI
```

## 2. Server Launch Flow

Let's trace exactly what happens when you run `python -m sglang.launch_server`.

### Step-by-Step Launch Sequence

```python
# sglang/srt/server.py — launch_server() (simplified)

def launch_server(server_args: ServerArgs):
    """Launch the SGLang server with all components."""
    
    # ============================================
    # PHASE 1: Configuration and Validation
    # ============================================
    
    # 1.1 Set up logging
    configure_logger(server_args)
    
    # 1.2 Validate arguments
    server_args.check_server_args()
    # Checks: valid model path, GPU availability, 
    # tp_size divides num_kv_heads, etc.
    
    # 1.3 Set environment variables
    os.environ["SGLANG_BLOCK_SIZE"] = str(server_args.page_size)
    # page_size is typically 1 (token-level) for RadixAttention
    
    # ============================================
    # PHASE 2: Create IPC Channels
    # ============================================
    
    # 2.1 Generate unique IPC addresses
    port_args = PortArgs.init_new(server_args)
    # Creates ZMQ IPC socket paths like:
    #   ipc:///tmp/sglang_tokenizer_<uuid>
    #   ipc:///tmp/sglang_scheduler_<uuid>
    #   ipc:///tmp/sglang_detokenizer_<uuid>
    
    # ============================================
    # PHASE 3: Launch Child Processes
    # ============================================
    
    # 3.1 Launch Scheduler process(es)
    # One scheduler per data-parallel rank
    scheduler_procs = []
    for dp_rank in range(server_args.dp_size):
        proc = mp.Process(
            target=run_scheduler_process,
            args=(server_args, port_args, dp_rank, 
                  server_args.tp_size,  # GPUs per scheduler
                  gpu_ids[dp_rank])     # which GPUs to use
        )
        proc.start()
        scheduler_procs.append(proc)
    # Inside run_scheduler_process:
    #   - Creates TpModelWorker(s) for tensor parallelism
    #   - Loads model weights onto GPU(s)
    #   - Creates RadixCache and MemoryPool
    #   - Enters scheduling event loop
    
    # 3.2 Launch TokenizerManager
    tokenizer_proc = mp.Process(
        target=run_tokenizer_process,
        args=(server_args, port_args)
    )
    tokenizer_proc.start()
    
    # 3.3 Launch DetokenizerManager
    detokenizer_proc = mp.Process(
        target=run_detokenizer_process,
        args=(server_args, port_args)
    )
    detokenizer_proc.start()
    
    # ============================================
    # PHASE 4: Wait for Ready
    # ============================================
    
    # 4.1 Wait for model loading to complete
    wait_for_model_ready(scheduler_procs)
    # This can take minutes for large models!
    
    # ============================================
    # PHASE 5: Start HTTP Server
    # ============================================
    
    # 5.1 Create FastAPI application
    app = create_app(server_args, port_args)
    
    # 5.2 Start uvicorn (blocking)
    uvicorn.run(
        app, 
        host=server_args.host, 
        port=server_args.port,
        log_level="warning"
    )
```

In [ ]:
# Simulate the server launch flow

import time
import uuid
from dataclasses import dataclass, field
from typing import List, Optional, Dict

@dataclass
class PortArgs:
    """Mirrors sglang.srt.utils.PortArgs"""
    tokenizer_ipc_name: str
    scheduler_input_ipc_name: str
    scheduler_output_ipc_name: str
    detokenizer_ipc_name: str
    
    @classmethod
    def init_new(cls, server_args):
        uid = str(uuid.uuid4())[:8]
        return cls(
            tokenizer_ipc_name=f"ipc:///tmp/sglang_tok_{uid}",
            scheduler_input_ipc_name=f"ipc:///tmp/sglang_sched_in_{uid}",
            scheduler_output_ipc_name=f"ipc:///tmp/sglang_sched_out_{uid}",
            detokenizer_ipc_name=f"ipc:///tmp/sglang_detok_{uid}",
        )

class ServerLaunchSimulator:
    """Simulates the SGLang server launch process."""
    
    def __init__(self, model_path, tp_size=1, dp_size=1):
        self.model_path = model_path
        self.tp_size = tp_size
        self.dp_size = dp_size
        self.log_entries = []
    
    def log(self, phase, message):
        self.log_entries.append((phase, message))
    
    def launch(self):
        # Phase 1: Configuration
        self.log("CONFIG", f"Model: {self.model_path}")
        self.log("CONFIG", f"TP size: {self.tp_size}, DP size: {self.dp_size}")
        self.log("CONFIG", f"Total GPUs needed: {self.tp_size * self.dp_size}")
        
        # Phase 2: IPC channels
        port_args = PortArgs.init_new(None)
        self.log("IPC", f"Tokenizer IPC: {port_args.tokenizer_ipc_name}")
        self.log("IPC", f"Scheduler IPC: {port_args.scheduler_input_ipc_name}")
        self.log("IPC", f"Detokenizer IPC: {port_args.detokenizer_ipc_name}")
        
        # Phase 3: Launch processes
        for dp_rank in range(self.dp_size):
            gpu_start = dp_rank * self.tp_size
            gpu_end = gpu_start + self.tp_size - 1
            self.log("LAUNCH", f"Scheduler #{dp_rank}: GPUs [{gpu_start}-{gpu_end}]")
            for tp_rank in range(self.tp_size):
                gpu_id = dp_rank * self.tp_size + tp_rank
                self.log("LAUNCH", f"  TpWorker (dp={dp_rank}, tp={tp_rank}): GPU {gpu_id}")
        
        self.log("LAUNCH", "TokenizerManager process started")
        self.log("LAUNCH", "DetokenizerManager process started")
        
        # Phase 4: Model loading
        self.log("LOAD", "Loading model weights...")
        self.log("LOAD", "Initializing RadixCache...")
        self.log("LOAD", "Allocating GPU memory pool...")
        self.log("LOAD", "Capturing CUDA graphs...")
        self.log("LOAD", "All processes ready!")
        
        # Phase 5: HTTP server
        self.log("HTTP", "FastAPI app created with OpenAI-compatible endpoints")
        self.log("HTTP", "Server listening on http://127.0.0.1:30000")
    
    def display(self):
        print("\nServer Launch Simulation")
        print("=" * 70)
        current_phase = None
        for phase, msg in self.log_entries:
            if phase != current_phase:
                print(f"\n--- Phase: {phase} ---")
                current_phase = phase
            print(f"  {msg}")

# Simulate single GPU launch
print("=== Single GPU Launch ===")
sim = ServerLaunchSimulator("meta-llama/Llama-3.1-8B-Instruct")
sim.launch()
sim.display()

print("\n\n=== Multi-GPU Launch (2 DP x 2 TP = 4 GPUs) ===")
sim2 = ServerLaunchSimulator("meta-llama/Llama-3.1-70B-Instruct", tp_size=2, dp_size=2)
sim2.launch()
sim2.display()

## 3. The FastAPI Server Layer

SGLang uses **FastAPI** for its HTTP server, providing OpenAI-compatible endpoints.

### 3.1 Endpoint Registration

```python
# sglang/srt/server.py — create_app() (simplified)

def create_app(server_args, port_args):
    app = FastAPI()
    
    # Create the tokenizer manager handle
    # (connects to TokenizerManager process via ZMQ)
    tokenizer_manager = TokenizerManager(
        server_args, port_args
    )
    
    # ---- OpenAI-Compatible Endpoints ----
    
    @app.post("/v1/chat/completions")
    async def chat_completions(request: ChatCompletionRequest):
        """OpenAI Chat Completions API."""
        # Convert OpenAI format to SGLang internal format
        gen_input = convert_chat_to_generate(request)
        
        if request.stream:
            return StreamingResponse(
                generate_stream(gen_input, tokenizer_manager)
            )
        else:
            result = await generate(gen_input, tokenizer_manager)
            return convert_to_openai_response(result)
    
    @app.post("/v1/completions")
    async def completions(request: CompletionRequest):
        """OpenAI Completions API."""
        gen_input = convert_completion_to_generate(request)
        result = await generate(gen_input, tokenizer_manager)
        return convert_to_openai_response(result)
    
    # ---- SGLang-Native Endpoints ----
    
    @app.post("/generate")
    async def sglang_generate(request: GenerateRequest):
        """SGLang native generation endpoint."""
        # Direct SGLang format (more flexible)
        result = await generate(request, tokenizer_manager)
        return result
    
    # ---- Health & Info Endpoints ----
    
    @app.get("/health")
    async def health():
        return {"status": "ok"}
    
    @app.get("/v1/models")
    async def list_models():
        return {"data": [{"id": server_args.model_path}]}
    
    @app.get("/get_server_info")
    async def get_server_info():
        return {
            "model_path": server_args.model_path,
            "tp_size": server_args.tp_size,
            "dp_size": server_args.dp_size,
            "schedule_policy": server_args.schedule_policy,
        }
    
    return app
```

In [ ]:
# Demonstrate the API endpoints and request/response formats

import json

# All endpoints provided by SGLang server
endpoints = [
    # OpenAI-compatible
    ("POST", "/v1/chat/completions", "Chat completion (OpenAI format)"),
    ("POST", "/v1/completions", "Text completion (OpenAI format)"),
    ("GET",  "/v1/models", "List available models"),
    ("POST", "/v1/embeddings", "Generate embeddings"),
    
    # SGLang-native
    ("POST", "/generate", "SGLang native generation"),
    ("POST", "/encode", "Encode text to tokens"),
    ("POST", "/decode", "Decode tokens to text"),
    
    # Health & Info
    ("GET",  "/health", "Health check"),
    ("GET",  "/health_generate", "Health check with generation test"),
    ("GET",  "/get_server_info", "Server configuration info"),
    ("GET",  "/get_memory_pool_size", "Current memory pool usage"),
    
    # Management
    ("POST", "/flush_cache", "Flush the RadixCache"),
    ("POST", "/update_weights", "Hot-reload model weights"),
]

print("SGLang Server Endpoints")
print("=" * 70)
print(f"{'Method':<8s} {'Path':<30s} {'Description'}")
print("-" * 70)
for method, path, desc in endpoints:
    print(f"{method:<8s} {path:<30s} {desc}")

# Example request/response for chat completions
print("\n\nExample: POST /v1/chat/completions")
print("=" * 50)

request_body = {
    "model": "meta-llama/Llama-3.1-8B-Instruct",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is 2+2?"}
    ],
    "max_tokens": 50,
    "temperature": 0.7,
    "top_p": 0.95,
}
print(f"\nRequest:\n{json.dumps(request_body, indent=2)}")

response_body = {
    "id": "chatcmpl-abc123",
    "object": "chat.completion",
    "model": "meta-llama/Llama-3.1-8B-Instruct",
    "choices": [{
        "index": 0,
        "message": {
            "role": "assistant",
            "content": "2 + 2 = 4."
        },
        "finish_reason": "stop"
    }],
    "usage": {
        "prompt_tokens": 25,
        "completion_tokens": 8,
        "total_tokens": 33
    }
}
print(f"\nResponse:\n{json.dumps(response_body, indent=2)}")

## 4. Request Flow Through the Runtime

### 4.1 The `generate()` Function

The core function that handles all generation requests:

```python
# sglang/srt/server.py — generate() (simplified)

async def generate(
    gen_input: GenerateReqInput,
    tokenizer_manager: TokenizerManager
):
    """Process a generation request.
    
    Flow:
    1. Send to TokenizerManager (tokenize text)
    2. TokenizerManager sends to Scheduler (schedule + execute)
    3. Scheduler sends output to DetokenizerManager (detokenize)
    4. DetokenizerManager returns text to caller
    """
    
    # Assign a unique request ID
    if gen_input.rid is None:
        gen_input.rid = str(uuid.uuid4())
    
    # Create an asyncio Event for this request
    event = asyncio.Event()
    state = RequestState(rid=gen_input.rid, event=event)
    
    # Register the request
    tokenizer_manager.register_request(state)
    
    # Send request to tokenizer process via ZMQ
    await tokenizer_manager.send_request(gen_input)
    
    # Wait for completion (the event is set when output arrives)
    await event.wait()
    
    # Return the result
    return state.result
```

In [ ]:
# Simulate the async request handling flow

import asyncio
from dataclasses import dataclass
from typing import Any
import uuid

@dataclass
class RequestState:
    """Tracks the state of an in-flight request."""
    rid: str
    result: Any = None
    finished: bool = False

class SimulatedTokenizerManager:
    """Simulates the TokenizerManager's role in request routing."""
    
    def __init__(self):
        self.pending_requests: Dict[str, RequestState] = {}
    
    def register_request(self, state: RequestState):
        self.pending_requests[state.rid] = state
        print(f"  [TokenizerMgr] Registered request {state.rid[:8]}...")
    
    def tokenize_and_forward(self, rid: str, text: str):
        # Simulate tokenization
        tokens = list(range(len(text.split())))
        print(f"  [TokenizerMgr] Tokenized '{text[:30]}...' -> {len(tokens)} tokens")
        print(f"  [TokenizerMgr] Forwarding to Scheduler via ZMQ")
        return tokens
    
    def receive_result(self, rid: str, result: str):
        state = self.pending_requests.get(rid)
        if state:
            state.result = result
            state.finished = True
            print(f"  [TokenizerMgr] Result received for {rid[:8]}...")

class SimulatedScheduler:
    """Simulates the Scheduler process."""
    
    def process_request(self, rid: str, tokens: list):
        print(f"  [Scheduler] Received {len(tokens)} tokens for {rid[:8]}...")
        print(f"  [Scheduler] RadixCache lookup: prefix match = 0 tokens")
        print(f"  [Scheduler] Allocating {len(tokens)} slots in memory pool")
        print(f"  [Scheduler] Adding to extend batch")
        
        # Simulate model execution
        output_tokens = [100, 200, 300, 400, 500]  # fake output
        print(f"  [Scheduler] Model forward: generated {len(output_tokens)} tokens")
        print(f"  [Scheduler] Sending output to DetokenizerManager")
        return output_tokens

class SimulatedDetokenizerManager:
    """Simulates DetokenizerManager."""
    
    def detokenize(self, tokens: list):
        result = f"[Simulated output from {len(tokens)} tokens]"
        print(f"  [DetokenizerMgr] Detokenized: '{result}'")
        return result

# Run the simulation
print("Complete Request Flow Simulation")
print("=" * 60)

# Create components
tok_mgr = SimulatedTokenizerManager()
scheduler = SimulatedScheduler()
detok_mgr = SimulatedDetokenizerManager()

# Simulate a request
rid = str(uuid.uuid4())
text = "What is the meaning of life? Please explain in detail."

print(f"\n1. HTTP Handler receives request: '{text[:40]}...'")
state = RequestState(rid=rid)

print(f"\n2. TokenizerManager processes:")
tok_mgr.register_request(state)
tokens = tok_mgr.tokenize_and_forward(rid, text)

print(f"\n3. Scheduler processes:")
output_tokens = scheduler.process_request(rid, tokens)

print(f"\n4. DetokenizerManager processes:")
result = detok_mgr.detokenize(output_tokens)

print(f"\n5. Result returned to HTTP handler:")
tok_mgr.receive_result(rid, result)
print(f"  Final result: '{state.result}'")
print(f"  Request completed: {state.finished}")

## 5. The Router: Request Distribution

### 5.1 Single Scheduler (Default)

With `dp_size=1`, all requests go to a single scheduler:

```python
# Simple routing: TokenizerManager -> single Scheduler
class TokenizerManager:
    def __init__(self, server_args, port_args):
        # Connect to scheduler via ZMQ
        self.scheduler_socket = zmq.Context().socket(zmq.PUSH)
        self.scheduler_socket.connect(port_args.scheduler_input_ipc_name)
    
    async def send_to_scheduler(self, tokenized_request):
        # Serialize and send via ZMQ
        self.scheduler_socket.send_pyobj(tokenized_request)
```

### 5.2 DataParallelController (dp_size > 1)

With data parallelism, a router distributes requests across multiple schedulers:

```python
# sglang/srt/managers/data_parallel_controller.py (simplified)

class DataParallelController:
    """Routes requests across data-parallel scheduler replicas."""
    
    def __init__(self, server_args, port_args):
        self.dp_size = server_args.dp_size
        
        # Connect to each scheduler replica
        self.scheduler_sockets = []
        for dp_rank in range(self.dp_size):
            socket = zmq.Context().socket(zmq.PUSH)
            socket.connect(port_args.get_scheduler_ipc(dp_rank))
            self.scheduler_sockets.append(socket)
        
        # Track load per scheduler
        self.request_counts = [0] * self.dp_size
    
    def route_request(self, request):
        """Route to least-loaded scheduler."""
        # Strategy: round-robin or least-loaded
        target = self._select_scheduler(request)
        self.scheduler_sockets[target].send_pyobj(request)
        self.request_counts[target] += 1
    
    def _select_scheduler(self, request):
        """Select which scheduler should handle this request."""
        # Option 1: Round-robin (simple)
        # return self.next_scheduler++ % self.dp_size
        
        # Option 2: Least loaded
        return min(range(self.dp_size), 
                   key=lambda i: self.request_counts[i])
        
        # Option 3: Cache-aware routing
        # Route to scheduler most likely to have prefix cached
        # (advanced, used in some configurations)
```

In [ ]:
# Simulate DataParallelController routing strategies

import random

class DataParallelRouter:
    """Simulates request routing across data-parallel schedulers."""
    
    def __init__(self, dp_size: int, strategy: str = "round_robin"):
        self.dp_size = dp_size
        self.strategy = strategy
        self.request_counts = [0] * dp_size
        self.rr_index = 0
        self.routing_log = []
    
    def route(self, request_id: str) -> int:
        """Route a request to a scheduler. Returns dp_rank."""
        if self.strategy == "round_robin":
            target = self.rr_index % self.dp_size
            self.rr_index += 1
        elif self.strategy == "least_loaded":
            target = min(range(self.dp_size), 
                        key=lambda i: self.request_counts[i])
        elif self.strategy == "random":
            target = random.randint(0, self.dp_size - 1)
        else:
            target = 0
        
        self.request_counts[target] += 1
        self.routing_log.append((request_id, target))
        return target
    
    def complete(self, dp_rank: int):
        """Mark a request as complete on a scheduler."""
        self.request_counts[dp_rank] = max(0, self.request_counts[dp_rank] - 1)

# Compare routing strategies
strategies = ["round_robin", "least_loaded", "random"]
dp_size = 4
num_requests = 20

print("Data Parallel Routing Strategy Comparison")
print(f"DP size: {dp_size}, Requests: {num_requests}")
print("=" * 60)

for strategy in strategies:
    random.seed(42)  # For reproducibility
    router = DataParallelRouter(dp_size, strategy)
    
    # Simulate routing with some completions
    for i in range(num_requests):
        router.route(f"req-{i}")
        # Simulate some requests completing
        if i > 5 and i % 3 == 0:
            router.complete(random.randint(0, dp_size - 1))
    
    # Count distribution
    distribution = [0] * dp_size
    for _, target in router.routing_log:
        distribution[target] += 1
    
    print(f"\n  Strategy: {strategy}")
    print(f"  Distribution: {distribution}")
    print(f"  Std Dev: {(sum((x - num_requests/dp_size)**2 for x in distribution) / dp_size)**0.5:.2f}")
    print(f"  Max imbalance: {max(distribution) - min(distribution)}")

## 6. Process Communication Deep Dive

### 6.1 ZMQ Socket Patterns Used

SGLang uses several ZMQ socket patterns:

| Pattern | Used For | Direction |
|---------|----------|----------|
| `PUSH/PULL` | TokenizerManager -> Scheduler | One-way |
| `PUSH/PULL` | Scheduler -> DetokenizerManager | One-way |
| `PUB/SUB` | Broadcast signals | One-to-many |
| `DEALER/ROUTER` | DP routing | Request/Reply |

### 6.2 Message Serialization

```python
# Messages are serialized using pickle (via zmq.send_pyobj)
# For large tensors, zero-copy is used

# Example: sending tokenized request
def send_tokenized_request(socket, request):
    # For small metadata: pickle
    metadata = {
        "rid": request.rid,
        "input_ids": request.input_ids,
        "sampling_params": request.sampling_params,
    }
    socket.send_pyobj(metadata)

# For multimodal (large pixel values):
def send_multimodal_request(socket, request):
    # Send metadata
    socket.send_pyobj(request.metadata, zmq.SNDMORE)
    # Send pixel values as bytes (zero-copy)
    socket.send(request.pixel_values.numpy().tobytes())
```

In [ ]:
# Demonstrate the message types flowing between processes

import json

message_types = {
    "FastAPI -> TokenizerManager": [
        {
            "type": "GenerateReqInput",
            "fields": {
                "rid": "req-abc123",
                "text": "What is AI?",
                "sampling_params": {"max_new_tokens": 50, "temperature": 0.7},
                "stream": False,
            }
        },
    ],
    "TokenizerManager -> Scheduler": [
        {
            "type": "TokenizedGenerateReqInput",
            "fields": {
                "rid": "req-abc123",
                "input_ids": [128000, 3923, 374, 15592, 30],
                "sampling_params": {"max_new_tokens": 50, "temperature": 0.7},
                "pixel_values": None,
                "image_sizes": None,
            }
        },
    ],
    "Scheduler -> DetokenizerManager": [
        {
            "type": "BatchTokenIDOut",
            "fields": {
                "rids": ["req-abc123"],
                "output_ids": [[15592, 374, 264, 5551, 2115]],
                "finished_reasons": ["stop"],
                "prompt_tokens": [5],
                "completion_tokens": [5],
            }
        },
    ],
    "DetokenizerManager -> FastAPI": [
        {
            "type": "BatchStrOut",
            "fields": {
                "rids": ["req-abc123"],
                "output_strs": ["AI is a broad field"],
                "finished": [True],
            }
        },
    ],
}

print("Message Types Between SGLang Processes")
print("=" * 60)

for channel, messages in message_types.items():
    print(f"\n--- {channel} ---")
    for msg in messages:
        print(f"  Type: {msg['type']}")
        for field, value in msg['fields'].items():
            val_str = str(value)
            if len(val_str) > 50:
                val_str = val_str[:50] + "..."
            print(f"    {field}: {val_str}")

## 7. The Scheduler Process Internals

The Scheduler is the **heart of SGLang's runtime**. Let's look at how it starts up and runs:

```python
# sglang/srt/managers/scheduler.py (simplified startup)

class Scheduler:
    def __init__(self, server_args, port_args, gpu_id, tp_rank, dp_rank):
        # 1. Create TpModelWorker (loads the model)
        self.tp_worker = TpModelWorker(
            server_args=server_args,
            gpu_id=gpu_id,
            tp_rank=tp_rank,
        )
        
        # 2. Get model config from worker
        self.model_config = self.tp_worker.model_config
        
        # 3. Create RadixCache for KV-cache management
        self.radix_cache = RadixCache(
            req_to_token_pool=self.tp_worker.req_to_token_pool,
            token_to_kv_pool=self.tp_worker.token_to_kv_pool,
        )
        
        # 4. Initialize scheduling data structures
        self.waiting_queue = []    # Requests waiting to be scheduled
        self.running_batch = None  # Currently executing batch
        
        # 5. Set up ZMQ sockets
        self.recv_from_tokenizer = zmq_socket(PULL, port_args.scheduler_input)
        self.send_to_detokenizer = zmq_socket(PUSH, port_args.detokenizer)
        
        # 6. Scheduling parameters
        self.schedule_policy = server_args.schedule_policy  # "lpm", "fcfs", etc.
        self.max_running_requests = server_args.max_running_requests
        self.max_total_tokens = self.tp_worker.max_total_num_tokens
    
    def event_loop(self):
        """Main scheduling loop — runs continuously."""
        while True:
            # 1. Receive new requests from TokenizerManager
            self.receive_requests()
            
            # 2. Build the next batch
            batch = self.get_next_batch()
            
            # 3. Execute model forward pass
            if batch is not None:
                output = self.tp_worker.forward_batch(batch)
                
                # 4. Process outputs (check EOS, update cache)
                self.process_batch_output(output)
                
                # 5. Send finished requests to detokenizer
                self.send_finished_requests()
```

In [ ]:
# Simulate the Scheduler's event loop

class SchedulerSimulation:
    """Simulates the Scheduler's main event loop."""
    
    def __init__(self, max_batch_size=8, max_total_tokens=4096):
        self.max_batch_size = max_batch_size
        self.max_total_tokens = max_total_tokens
        self.waiting_queue = []
        self.running_requests = []
        self.completed_requests = []
        self.current_tokens = 0
        self.step = 0
    
    def add_request(self, rid: str, num_input_tokens: int, max_output_tokens: int):
        self.waiting_queue.append({
            "rid": rid,
            "input_tokens": num_input_tokens,
            "max_output_tokens": max_output_tokens,
            "generated_tokens": 0,
            "status": "waiting",
        })
    
    def schedule_step(self):
        """One iteration of the scheduler event loop."""
        self.step += 1
        print(f"\n--- Scheduler Step {self.step} ---")
        
        # 1. Move waiting requests to running if capacity allows
        newly_scheduled = []
        while (self.waiting_queue and 
               len(self.running_requests) < self.max_batch_size):
            req = self.waiting_queue.pop(0)
            req["status"] = "running"
            self.running_requests.append(req)
            newly_scheduled.append(req["rid"])
        
        if newly_scheduled:
            print(f"  Scheduled: {newly_scheduled}")
        
        # 2. Execute forward pass for running requests
        if not self.running_requests:
            print("  No requests to process")
            return
        
        extend_reqs = [r for r in self.running_requests if r["generated_tokens"] == 0]
        decode_reqs = [r for r in self.running_requests if r["generated_tokens"] > 0]
        
        if extend_reqs:
            print(f"  Extend batch: {len(extend_reqs)} requests (prefill)")
        if decode_reqs:
            print(f"  Decode batch: {len(decode_reqs)} requests")
        
        # 3. Generate one token per request
        finished = []
        for req in self.running_requests:
            req["generated_tokens"] += 1
            if req["generated_tokens"] >= req["max_output_tokens"]:
                finished.append(req)
        
        # 4. Move finished requests
        for req in finished:
            req["status"] = "completed"
            self.running_requests.remove(req)
            self.completed_requests.append(req)
            print(f"  Completed: {req['rid']} ({req['generated_tokens']} tokens)")
        
        print(f"  Status: waiting={len(self.waiting_queue)}, "
              f"running={len(self.running_requests)}, "
              f"completed={len(self.completed_requests)}")

# Run simulation
scheduler = SchedulerSimulation(max_batch_size=3)

# Add some requests
scheduler.add_request("req-A", 100, 3)  # Short output
scheduler.add_request("req-B", 50, 5)   # Medium output
scheduler.add_request("req-C", 200, 2)  # Very short output
scheduler.add_request("req-D", 75, 4)   # Waiting (over batch limit)
scheduler.add_request("req-E", 150, 3)  # Waiting

print("Scheduler Event Loop Simulation")
print(f"Max batch size: {scheduler.max_batch_size}")
print(f"Total requests: {len(scheduler.waiting_queue)}")

# Run for several steps
for _ in range(8):
    scheduler.schedule_step()
    if not scheduler.waiting_queue and not scheduler.running_requests:
        print("\nAll requests completed!")
        break

## 8. Streaming Response Support

SGLang supports **Server-Sent Events (SSE)** for streaming responses:

```python
# sglang/srt/server.py — streaming support (simplified)

async def generate_stream(
    gen_input: GenerateReqInput,
    tokenizer_manager: TokenizerManager
):
    """Generate with streaming output."""
    gen_input.stream = True
    
    # Register request
    rid = gen_input.rid
    event_queue = asyncio.Queue()  # For streaming chunks
    tokenizer_manager.register_stream(rid, event_queue)
    
    # Send request
    await tokenizer_manager.send_request(gen_input)
    
    # Yield chunks as they arrive
    while True:
        chunk = await event_queue.get()
        
        if chunk.finished:
            # Send final chunk
            yield format_sse_event(chunk)
            yield "data: [DONE]\n\n"
            break
        else:
            # Send incremental chunk
            yield format_sse_event(chunk)

def format_sse_event(chunk):
    """Format as Server-Sent Event."""
    data = {
        "id": chunk.rid,
        "object": "chat.completion.chunk",
        "choices": [{
            "delta": {"content": chunk.text},
            "finish_reason": chunk.finish_reason,
        }]
    }
    return f"data: {json.dumps(data)}\n\n"
```

In [ ]:
# Simulate SSE streaming

import time

def simulate_streaming_response():
    """Simulate what an SSE stream looks like."""
    chunks = [
        # (text, finish_reason)
        ("The", None),
        (" capital", None),
        (" of", None),
        (" France", None),
        (" is", None),
        (" Paris", None),
        (".", "stop"),
    ]
    
    print("SSE Streaming Response")
    print("=" * 60)
    print("Content-Type: text/event-stream")
    print()
    
    accumulated = ""
    for text, finish_reason in chunks:
        accumulated += text
        event = {
            "id": "chatcmpl-abc123",
            "object": "chat.completion.chunk",
            "choices": [{
                "index": 0,
                "delta": {"content": text},
                "finish_reason": finish_reason
            }]
        }
        print(f"data: {json.dumps(event)}")
        print()  # Empty line between events
    
    print("data: [DONE]")
    print()
    print(f"\nAccumulated text: '{accumulated}'")

simulate_streaming_response()

## 9. Engine API (Offline Inference)

Besides the HTTP server, SGLang provides an **Engine API** for offline (in-process) inference:

```python
# sglang/api.py — Engine class (simplified)

class Engine:
    """Offline inference engine. No HTTP server needed."""
    
    def __init__(self, model_path, tp_size=1, **kwargs):
        # Create ServerArgs
        self.server_args = ServerArgs(
            model_path=model_path,
            tp_size=tp_size,
            **kwargs
        )
        
        # Launch processes (same as server, but no HTTP)
        self._launch_processes()
    
    def generate(
        self, 
        prompt, 
        sampling_params=None,
    ):
        """Generate text for one or more prompts."""
        if isinstance(prompt, str):
            prompts = [prompt]
        else:
            prompts = prompt
        
        # Send all prompts to the runtime
        results = []
        for p in prompts:
            req = GenerateReqInput(
                text=p,
                sampling_params=sampling_params or {},
            )
            result = self._process_request(req)
            results.append(result)
        
        return results if len(results) > 1 else results[0]
    
    def shutdown(self):
        """Clean up all processes."""
        for proc in self.processes:
            proc.terminate()
            proc.join()
```

In [ ]:
# Demonstrate the difference between Server and Engine modes

print("SGLang Usage Modes")
print("=" * 60)

modes = {
    "Server Mode (HTTP)": {
        "launch": "python -m sglang.launch_server --model-path <model>",
        "client": "curl, openai.Client, requests",
        "use_case": "Production serving, multi-client",
        "process_model": "HTTP server + worker processes",
        "code": '''# Server side
python -m sglang.launch_server --model-path meta-llama/Llama-3.1-8B-Instruct --port 30000

# Client side
import openai
client = openai.Client(base_url="http://localhost:30000/v1", api_key="none")
response = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[{"role": "user", "content": "Hello!"}]
)''',
    },
    "Engine Mode (Offline)": {
        "launch": "sgl.Engine(model_path=<model>)",
        "client": "Direct Python API",
        "use_case": "Batch processing, scripts, notebooks",
        "process_model": "Worker processes only (no HTTP)",
        "code": '''import sglang as sgl

engine = sgl.Engine(model_path="meta-llama/Llama-3.1-8B-Instruct")
result = engine.generate("Hello!", sampling_params={"max_new_tokens": 50})
print(result["text"])
engine.shutdown()''',
    },
    "Runtime Mode (DSL)": {
        "launch": "sgl.Runtime(model_path=<model>)",
        "client": "SGLang DSL functions",
        "use_case": "Complex LLM programs, multi-step reasoning",
        "process_model": "Full server (in background)",
        "code": '''import sglang as sgl

runtime = sgl.Runtime(model_path="meta-llama/Llama-3.1-8B-Instruct")
sgl.set_default_backend(runtime)

@sgl.function
def chat(s, question):
    s += sgl.user(question)
    s += sgl.assistant(sgl.gen("answer", max_tokens=100))

state = chat.run(question="Hello!")
print(state["answer"])
runtime.shutdown()''',
    },
}

for mode_name, info in modes.items():
    print(f"\n{'=' * 60}")
    print(f"  {mode_name}")
    print(f"{'=' * 60}")
    print(f"  Launch: {info['launch']}")
    print(f"  Client: {info['client']}")
    print(f"  Use case: {info['use_case']}")
    print(f"  Process model: {info['process_model']}")
    print(f"\n  Code:")
    for line in info['code'].split('\n'):
        print(f"    {line}")

## 10. Error Handling and Recovery

SGLang has several error handling mechanisms:

```python
# Error handling in the runtime

class Scheduler:
    def event_loop(self):
        while True:
            try:
                # Normal scheduling loop
                self.receive_requests()
                batch = self.get_next_batch()
                if batch:
                    output = self.tp_worker.forward_batch(batch)
                    self.process_batch_output(output)
            except torch.cuda.OutOfMemoryError:
                # OOM: abort some requests and retry
                logger.error("CUDA OOM! Aborting requests...")
                self.abort_running_requests()
                torch.cuda.empty_cache()
            except Exception as e:
                # Unknown error: log and continue
                logger.error(f"Scheduler error: {e}")
                traceback.print_exc()
                # Abort affected requests with error
                self.abort_with_error(str(e))

# Process monitoring
def monitor_processes(procs):
    """Watch for process crashes."""
    for proc in procs:
        if not proc.is_alive():
            logger.error(f"Process {proc.name} crashed!")
            # Could restart or abort server
            raise RuntimeError(f"Process {proc.name} died")
```

In [ ]:
# Demonstrate request timeout and error handling

class RequestErrorHandler:
    """Handles various error scenarios in SGLang runtime."""
    
    ERROR_SCENARIOS = {
        "CUDA OOM": {
            "cause": "Too many concurrent requests exceed GPU memory",
            "recovery": "Abort lowest-priority requests, clear CUDA cache",
            "prevention": "Set --mem-fraction-static lower, reduce batch size",
        },
        "Request Timeout": {
            "cause": "Request exceeds maximum generation time",
            "recovery": "Cancel request, return timeout error to client",
            "prevention": "Set appropriate max_tokens, increase timeout",
        },
        "Invalid Input": {
            "cause": "Input exceeds context length or contains invalid tokens",
            "recovery": "Return 400 error immediately (before scheduling)",
            "prevention": "Validate input length on client side",
        },
        "Model Error": {
            "cause": "NaN in model output, corrupted weights",
            "recovery": "Abort batch, log error, return 500",
            "prevention": "Use validated model weights, check dtype",
        },
        "Process Crash": {
            "cause": "Worker process segfault, CUDA error",
            "recovery": "Server must restart (processes are not auto-restarted)",
            "prevention": "Monitor with health checks, use Docker restart policy",
        },
    }
    
    @classmethod
    def display(cls):
        print("SGLang Error Handling Scenarios")
        print("=" * 70)
        for error, info in cls.ERROR_SCENARIOS.items():
            print(f"\n  Error: {error}")
            print(f"    Cause:      {info['cause']}")
            print(f"    Recovery:   {info['recovery']}")
            print(f"    Prevention: {info['prevention']}")

RequestErrorHandler.display()

## 11. Performance Monitoring

SGLang provides several endpoints for monitoring performance:

```python
# Monitoring endpoints

@app.get("/get_server_info")
async def get_server_info():
    return {
        "model_path": server_args.model_path,
        "tp_size": server_args.tp_size,
        "schedule_policy": server_args.schedule_policy,
        "current_running": scheduler.num_running,
        "current_waiting": scheduler.num_waiting,
        "cache_hit_rate": scheduler.radix_cache.hit_rate(),
        "gpu_memory_used": torch.cuda.memory_allocated(),
    }

@app.get("/get_memory_pool_size")
async def get_memory_pool_size():
    return {
        "total_slots": memory_pool.total_size,
        "used_slots": memory_pool.used_size,
        "available_slots": memory_pool.available_size,
    }
```

In [ ]:
# Simulate monitoring dashboard

class ServerMonitor:
    """Simulates SGLang server monitoring."""
    
    def __init__(self):
        self.metrics_history = []
    
    def collect_metrics(self, step):
        """Simulate collecting server metrics."""
        import random
        random.seed(step)
        
        metrics = {
            "step": step,
            "running_requests": random.randint(5, 32),
            "waiting_requests": random.randint(0, 10),
            "cache_hit_rate": 0.3 + random.random() * 0.5,
            "tokens_per_second": 2000 + random.randint(-200, 500),
            "gpu_memory_pct": 75 + random.random() * 15,
            "avg_latency_ms": 50 + random.random() * 100,
            "batch_size": random.randint(8, 32),
        }
        self.metrics_history.append(metrics)
        return metrics
    
    def display_dashboard(self):
        """Display monitoring dashboard."""
        print("\nSGLang Server Dashboard")
        print("=" * 80)
        print(f"{'Step':>5s} {'Running':>8s} {'Waiting':>8s} {'Cache%':>8s} "
              f"{'Tok/s':>8s} {'GPU%':>8s} {'Lat(ms)':>8s} {'Batch':>6s}")
        print("-" * 80)
        
        for m in self.metrics_history:
            print(f"{m['step']:5d} "
                  f"{m['running_requests']:8d} "
                  f"{m['waiting_requests']:8d} "
                  f"{m['cache_hit_rate']:7.1%} "
                  f"{m['tokens_per_second']:8d} "
                  f"{m['gpu_memory_pct']:7.1f}% "
                  f"{m['avg_latency_ms']:8.1f} "
                  f"{m['batch_size']:6d}")

# Collect metrics over time
monitor = ServerMonitor()
for step in range(1, 11):
    monitor.collect_metrics(step)

monitor.display_dashboard()

## 12. Summary

### Key Takeaways

1. **Multi-process architecture**: TokenizerManager, Scheduler, DetokenizerManager run as separate processes connected via ZMQ
2. **FastAPI server**: Provides OpenAI-compatible endpoints plus SGLang-native endpoints
3. **DataParallelController**: Routes requests across multiple scheduler replicas for higher throughput
4. **Three usage modes**: Server (HTTP), Engine (offline), Runtime (DSL)
5. **Streaming**: SSE-based streaming for real-time token delivery
6. **The Scheduler event loop** is the heart of the runtime, managing request lifecycle

### Next Chapter

Chapter 5.3 will dive deep into **RadixAttention** — the radix tree data structure that enables SGLang's efficient KV-cache reuse.

## Exercises

### Exercise 1: Custom Route
Add a custom `/v1/tokenize` endpoint that accepts text and returns token IDs and token count. Write the FastAPI route handler.

### Exercise 2: Load Balancing
Implement a "cache-aware" routing strategy for DataParallelController that routes requests to the scheduler most likely to have a matching prefix in its RadixCache.

### Exercise 3: Monitoring
Design a Prometheus metrics integration for SGLang. What metrics would you expose? Write the metric definitions.

### Exercise 4: Trace Analysis
Given the request lifecycle trace from Section 4, identify which steps are:
- CPU-bound
- GPU-bound  
- I/O-bound
Which step is the bottleneck for (a) short prompts and (b) long prompts?

In [ ]:
# Exercise 1 Starter Code

print("""# Exercise 1: Add a custom tokenize endpoint

from fastapi import FastAPI
from pydantic import BaseModel
from typing import List

class TokenizeRequest(BaseModel):
    text: str
    model: str = "default"

class TokenizeResponse(BaseModel):
    tokens: List[int]
    count: int

@app.post("/v1/tokenize")
async def tokenize(request: TokenizeRequest) -> TokenizeResponse:
    # TODO: Implement this endpoint
    # 1. Get the tokenizer from tokenizer_manager
    # 2. Tokenize the input text
    # 3. Return token IDs and count
    pass
""")